# 00 — deepCab architectural tour

**Goal:** walk through where each piece lives, how data flows, and the design decisions that make the agent + API + Prefect flow all consume the same Pydantic schemas without drift.

**Prereqs:** `uv sync --extra dev` and an active `.env.dev`. No services required for this notebook — it's introspective.


## Package layout

```
deepCab/
├── schemas/      Pydantic models — single source of truth
├── data/         Polars + DuckDB + Parquet/Hive + Pandera + lineage SQLite
├── features/     Stateless transforms (Polars) + ColumnTransformer assembly
├── models/       AbstractEstimator + 6 backends + ACI + ONNX export
├── training/     Pure functions: preprocess / train / evaluate / predict / cv / hpo / provenance
├── explain/      SHAP factory + 65d→5-group aggregation + cached summary
├── registry/     Backend-agnostic save/load + MLflow aliases + auto MODEL_CARD.md
├── serving/      ONNX runtime + async batcher + INT8 quant
├── api/          FastAPI factory + routers + lifespan + middleware + DI
├── agent/        Tool registry + planner + executor + budget + improve loop
├── obs/          JSONL tracer + structlog + OTel + Prom registries
└── flow_v2/      Prefect 3 retrain flow + schedules
```


In [ ]:
# Confirm the import surface is what we expect.
from deepCab.schemas.config import BackendConfig, TrainConfig
from deepCab.models import build_estimator
from deepCab.agent import openai_tools, tool_names
from deepCab.training.train import run as train_run

print(f"Registered agent tools: {len(tool_names())}")
print(f"Tool names: {tool_names()}")

## Single source of truth — Pydantic everywhere

The `BackendConfig` discriminated union is consumed by **four** subsystems with zero duplication:

1. **FastAPI** — `POST /train` accepts a body whose schema is generated from `TrainConfig`.
2. **Hydra** — `python -m deepCab.training.train backend=xgb` resolves to the same TrainConfig via `OmegaConf.to_container` → `TrainConfig.model_validate`.
3. **OpenAI agent tools** — `openai_tools()` emits the function-calling schema list directly from `InputModel.model_json_schema()`.
4. **MLflow params** — `model_dump(mode='json')` is logged on every run (flattened to dotted keys).


In [ ]:
from deepCab.schemas.config import LGBMConfig

cfg = LGBMConfig(n_estimators=200, num_leaves=63)

# Same schema → API body shape:
import json

print("JSON schema (first 400 chars):")
print(json.dumps(LGBMConfig.model_json_schema(), indent=2)[:400], "...")

# Same instance → MLflow flat params:
from deepCab.training._mlflow import flatten

print("\nFlat MLflow params:")
print(flatten(cfg.model_dump(mode="json")))

## Data flow

```
Parquet (Hive: dataset_size/year/month)
  └─ data.io.scan → Polars LazyFrame
       └─ training.preprocess.clean (NYC bounds, fare/pax filters)
            └─ features.pipeline.preprocess_features → (N, 65) numpy
                 └─ build_estimator(cfg.backend).fit(X, y, validation_data)
                      └─ ACIRegressor.from_fitted(estimator, X_calib, y_calib)
                           └─ STATE.set_model(ModelHandle(estimator, aci, background))
                                ├─ provenance.json + MODEL_CARD.md → runs/<run_id>/
                                ├─ LineageEdge → SQLite at REGISTRY_LOCAL_PATH/lineage.db
                                └─ ONNX export + RuntimeRegistry.register
```

Every entry point — Hydra CLI, Prefect flow, agent `train` tool — flows through `training.train.run(cfg)` which is the **only** code path that mutates STATE + emits artifacts. Single funnel = no double-fit, no drift.


## Where to look next

- `02-backend-zoo.ipynb` — the discriminated-union factory in action.
- `07-agent-loop.ipynb` — `make_plan` + `run_one_turn` with a stubbed OpenAI client.
- `CLAUDE.md` (parent dir) — per-module tree + per-phase rationale.
